# EDA Q5

`code/5_make_figures.py`에서 `5_vif_bar`, `5_batch_correlation_heatmap`만 추려 실행하는 노트북입니다.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio
from IPython.display import Image, display

try:
    import mat73
except ImportError:
    mat73 = None


warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 11


PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output" / "img"
MAT_FILES = sorted(DATA_DIR.glob("*.mat"))

SUMMARY_COLUMN_MAP = {
    "QD": "QDischarge",
    "QC": "QCharge",
    "IR": "IR",
    "Tmax": "Tmax",
    "Tavg": "Tavg",
    "Tmin": "Tmin",
    "chargetime": "chargetime",
}


def load_mat(path: Path):
    last_error = None
    if mat73 is not None:
        try:
            return mat73.loadmat(str(path)), "mat73"
        except Exception as exc:
            last_error = exc
    try:
        return sio.loadmat(str(path), simplify_cells=True), "scipy.io.loadmat"
    except Exception as exc:
        if last_error is not None:
            raise RuntimeError(f"Failed to load {path.name}: {last_error} | {exc}")
        raise RuntimeError(f"Failed to load {path.name}: {exc}")


def scalarize(value):
    if isinstance(value, np.ndarray):
        if value.shape == ():
            return value.item()
        if value.size == 1:
            return value.reshape(-1)[0].item()
    if hasattr(value, "item") and not isinstance(value, (str, bytes)):
        try:
            return value.item()
        except Exception:
            pass
    return value


def normalize_records(obj):
    if isinstance(obj, list):
        return obj
    if isinstance(obj, tuple):
        return list(obj)
    if isinstance(obj, np.ndarray):
        return list(obj)
    if isinstance(obj, dict):
        keys = list(obj.keys())
        if not keys:
            return []
        n = len(obj[keys[0]])
        return [{k: obj[k][i] for k in keys} for i in range(n)]
    raise TypeError(f"Unsupported container: {type(obj)}")


def to_1d_array(value):
    arr = np.asarray(value)
    if arr.shape == ():
        arr = arr.reshape(1)
    return arr.reshape(-1)


def batch_name_from_path(path: Path) -> str:
    name = path.stem
    suffix = "_batchdata_updated_struct_errorcorrect"
    return name.replace(suffix, "") if suffix in name else name


def extract_summary_from_batch(batch, batch_name: str, file_name: str):
    rows = []
    batch = normalize_records(batch)

    for cell_idx, cell in enumerate(batch):
        summary = cell.get("summary")
        if not isinstance(summary, dict):
            continue

        arrays = {}
        lengths = []
        for out_col, src_col in SUMMARY_COLUMN_MAP.items():
            if src_col not in summary:
                arrays[out_col] = None
                continue
            arr = to_1d_array(summary[src_col])
            arrays[out_col] = arr
            lengths.append(len(arr))

        if not lengths:
            continue

        n_cycles = min(lengths)
        cycle_life = scalarize(cell.get("cycle_life"))
        policy = cell.get("policy_readable") or cell.get("policy") or "unknown"
        cell_uid = f"{batch_name}_cell_{cell_idx:03d}"

        for cycle_idx in range(n_cycles):
            row = {
                "file_name": file_name,
                "batch_name": batch_name,
                "cell_uid": cell_uid,
                "cycle": cycle_idx + 1,
                "cycle_life": cycle_life,
                "charging_policy": str(policy),
            }
            for out_col, arr in arrays.items():
                row[out_col] = np.nan if arr is None else arr[cycle_idx]
            rows.append(row)
    return rows


def compute_vif(df: pd.DataFrame) -> pd.DataFrame:
    values = []
    X = df.to_numpy(dtype=float)
    for i, col in enumerate(df.columns):
        y = X[:, i]
        others = np.delete(X, i, axis=1)
        others = np.column_stack([np.ones(len(others)), others])
        beta, *_ = np.linalg.lstsq(others, y, rcond=None)
        pred = others @ beta
        ss_res = np.sum((y - pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot else 1.0
        vif = np.inf if r2 >= 1 else 1 / (1 - r2)
        values.append({"feature": col, "r2_from_others": r2, "vif": vif})
    return pd.DataFrame(values).sort_values("vif", ascending=False)


def build_early_features() -> pd.DataFrame:
    rows = []
    for path in MAT_FILES:
        data, loader = load_mat(path)
        batch_name = batch_name_from_path(path)
        batch_rows = extract_summary_from_batch(data["batch"], batch_name, path.name)
        rows.extend(batch_rows)
        print(f"loaded {path.name:<55} -> rows={len(batch_rows):>7,} via {loader}")

    df = pd.DataFrame(rows)
    zero_signal_cols = ["QD", "QC", "IR", "Tmax", "Tavg", "Tmin", "chargetime"]
    zero_mask = df[zero_signal_cols].fillna(0).eq(0).all(axis=1)

    clean = df.loc[~zero_mask].copy()
    clean = clean.loc[clean["cycle_life"].notna()].copy()

    early = (
        clean.loc[clean["cycle"] <= 100]
        .groupby(["batch_name", "cell_uid"])
        .agg(
            cycle_life=("cycle_life", "first"),
            mean_QD=("QD", "mean"),
            std_QD=("QD", "std"),
            mean_QC=("QC", "mean"),
            mean_IR=("IR", "mean"),
            mean_Tavg=("Tavg", "mean"),
            mean_Tmax=("Tmax", "mean"),
            mean_Tmin=("Tmin", "mean"),
            mean_chargetime=("chargetime", "mean"),
        )
        .reset_index()
    )
    return early


def save_batch_correlation_heatmap(early: pd.DataFrame, feature_cols):
    batches = sorted(early["batch_name"].unique())
    matrix = []
    for batch in batches:
        corr = (
            early.loc[early["batch_name"] == batch, feature_cols + ["cycle_life"]]
            .corr(numeric_only=True)["cycle_life"]
            .drop("cycle_life")
            .reindex(feature_cols)
        )
        matrix.append(corr.values)

    corr_df = pd.DataFrame(matrix, index=batches, columns=feature_cols)

    fig, ax = plt.subplots(figsize=(11, 4))
    im = ax.imshow(corr_df, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(feature_cols)))
    ax.set_yticks(range(len(batches)))
    ax.set_xticklabels(feature_cols, rotation=45, ha="right")
    ax.set_yticklabels(batches)

    for i in range(len(batches)):
        for j in range(len(feature_cols)):
            value = corr_df.iloc[i, j]
            color = "white" if abs(value) > 0.5 else "black"
            ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=9, color=color)

    ax.set_title("Batch-wise Correlation With Cycle Life")
    plt.tight_layout()
    out = OUTPUT_DIR / "5_batch_correlation_heatmap.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out, corr_df


def save_vif_bar(early: pd.DataFrame, feature_cols):
    vif_table = compute_vif(early[feature_cols].dropna())

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(vif_table["feature"], vif_table["vif"], color="slateblue", alpha=0.85)
    ax.axhline(5, color="darkorange", linestyle="--", linewidth=1.5, label="VIF = 5")
    ax.axhline(10, color="tomato", linestyle="--", linewidth=1.5, label="VIF = 10")
    ax.set_title("Multicollinearity Check: VIF")
    ax.set_ylabel("VIF")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
    plt.tight_layout()
    out = OUTPUT_DIR / "5_vif_bar.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out, vif_table


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

feature_cols = [
    "mean_QD",
    "std_QD",
    "mean_QC",
    "mean_IR",
    "mean_Tavg",
    "mean_Tmax",
    "mean_Tmin",
    "mean_chargetime",
]

early = build_early_features()
heatmap_path, corr_df = save_batch_correlation_heatmap(early, feature_cols)
vif_path, vif_table = save_vif_bar(early, feature_cols)

display(Image(filename=str(heatmap_path)))
display(Image(filename=str(vif_path)))

corr_df.round(3)


In [ ]:
vif_table.round(3)